In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.llms import Cohere
from langchain_cohere import CohereRerank


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [6]:
# Carrega modelos Open AI
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')
llm = ChatOpenAI(model_name='gpt-3.5-turbo', max_tokens=300)


In [7]:
# Carrega o PDF
pdf_link = 'Clean Architecture.pdf'

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()

In [ ]:
# Separa em Chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 4000,
    chunk_overlap = 20,
    length_function = len,
    add_start_index = True
)

chunks = text_splitter.split_documents(pages)


In [11]:
# Salva os chunks no vector DB

vectordb = Chroma(embedding_function=embeddings_model, persist_directory='db')

In [12]:
# Carrega os documentos do DB para rankeamento, 
# exigindo mais documentos ter mais margem para otimizar o rankeamento

naive_retriever = vectordb.as_retriever(search_kwargs={"k": 10})

In [ ]:
os.environ["COHERE_API_KEY"] = ""

In [15]:
# Retorna os top 3 documentos após o rerank dos 10 primeiros documentos selecionados

rerank = CohereRerank(top_n=3, model='rerank-v3.5')

compression_retriever = ContextualCompressionRetriever(
    base_compressor=rerank,
    base_retriever=naive_retriever
)

In [19]:
TEMPLATE = """""
    Você é um arquiteto de sistemas experiente. Responda a pergunta utilizando o contexto informado.

    Query:
    {question}

    Context:
    {context}
    
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [21]:
setup_retrieval = RunnableParallel({"question": RunnablePassthrough(), "context": compression_retriever})

output_parser = StrOutputParser()

compressor_retrieval_chain = setup_retrieval | rag_prompt | llm | output_parser

In [22]:
compressor_retrieval_chain.invoke("Como a camada Application trata o domínio na Clean Architecture")

'Na Clean Architecture, a camada de Application é responsável por orquestrar as regras de negócio e coordenar a interação entre as camadas de Domínio e Infraestrutura. Ela atua como um intermediário entre a camada de Domínio, que contém as entidades e regras de negócio da aplicação, e a camada de Infraestrutura, que lida com detalhes de implementação como banco de dados, APIs externas, etc. A camada de Application é onde são implementados os casos de uso da aplicação, manipulando os objetos do domínio de acordo com as regras de negócio definidas.'